# Pattern Detection – Day 6: Data & Domain

**Student Name:** Traver Dinten
**Country:** Switzerland
**Semester term:** FS26

## Use Case
*Focus: domain and application context*

In the context of satellite-based glacier surface analysis, Sentinel-2 imagery of the Aletsch Glacier is used to identify structural patterns such as moraine lines, medial moraines, and ice-rock boundaries. These linear features are analyzed by glaciologists to assess glacier dynamics, flow patterns, and retreat behaviour. Detecting linear structures in satellite imagery is particularly relevant for Switzerland because changes in moraine positions and glacier boundary geometry serve as key indicators of glacier instability and retreat, which has direct implications for downstream water resources and Alpine infrastructure safety.

## Problem Statement
*Focus: technical vulnerability*

This project addresses the problem of applying edge detection to Sentinel-2 satellite imagery of glaciers within the context of structural feature monitoring. At the 10 m pixel resolution of Sentinel-2, structural features such as moraine lines and glacier boundaries appear as subtle intensity transitions spanning only a few pixels. If edge detection parameters — specifically blur kernel size and threshold values — are chosen inadequately, the result may either miss critical structural edges (under-detection) or produce excessive false edges from surface texture variations (over-detection), leading to unreliable structural maps.

## Experimental Objective
*Focus: investigation goal*

The objective of this project is to investigate how the parameterization of Canny edge detection influences the detection quality of linear structures (moraine lines, glacier boundaries) in Sentinel-2 satellite imagery of the Aletsch Glacier tongue. The goal is to determine which parameter configurations yield edge maps that reliably capture geologically meaningful structures while suppressing noise.

## Data Definition, Source, and Visualization

The selected image is a Sentinel-2 Level-2A Surface Reflectance composite of the Aletsch Glacier tongue region, acquired during summer 2023 (June–September). The composite uses bands B4 (Red), B3 (Green), B2 (Blue) at 10 m resolution. The glacier tongue is chosen because it exhibits clearly visible linear features: medial moraines running along the glacier flow, lateral moraine boundaries, and the glacier terminus boundary — all suitable targets for edge detection. The data originate from Copernicus Sentinel-2 via Google Earth Engine (`COPERNICUS/S2_SR_HARMONIZED`), freely available under the Copernicus open data policy.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join("..", "helpers"))

import numpy as np
import cv2
import matplotlib.pyplot as plt

DATA_DIR = os.path.join("..", "data", "raw")
FIGURES_DIR = os.path.join("..", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

IMAGE_PATH = os.path.join(DATA_DIR, "aletsch_tongue_s2.tif")

# Download via GEE only if not already cached
if not os.path.exists(IMAGE_PATH):
    from gee_utils import (
        init_gee, get_sentinel2_composite, download_sentinel2,
        ALETSCH_TONGUE_BBOX,
    )
    init_gee("gbsvmc2")
    composite = get_sentinel2_composite(bbox=ALETSCH_TONGUE_BBOX)
    download_sentinel2(composite, ALETSCH_TONGUE_BBOX, IMAGE_PATH)
else:
    print(f"Image already cached: {IMAGE_PATH}")

from gee_utils import load_image
img_rgb = load_image(IMAGE_PATH)
print(f"Image shape: {img_rgb.shape}")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
ax.imshow(img_rgb)
ax.set_title("Sentinel-2 – Aletsch Glacier Tongue")
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "day06_original.png"), dpi=150, bbox_inches="tight")
plt.show()

**Observations:**
The Sentinel-2 composite shows the Aletsch Glacier tongue with visible structural features: dark medial moraine lines running along the glacier flow direction, contrast boundaries between glacier ice and surrounding rock/vegetation, and the glacier terminus area. At 10 m resolution, these features appear as relatively narrow intensity transitions, presenting a challenging scenario for edge detection where true structural edges must be distinguished from surface texture variations.

## Algorithmic Pipeline Preview

The Canny edge detection pipeline consists of several stages. Below we visualize the key intermediate steps to establish the processing chain:
1. RGB → Grayscale conversion
2. Gaussian blur (noise suppression)
3. Canny edge detection (gradient + non-maximum suppression + hysteresis thresholding)

In [ ]:
gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
blurred = cv2.GaussianBlur(gray, (0, 0), sigmaX=1.0)
edges = cv2.Canny(blurred, 50, 150)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(img_rgb)
axes[0].set_title("1. Original (RGB)")
axes[0].axis("off")

axes[1].imshow(gray, cmap="gray")
axes[1].set_title("2. Grayscale")
axes[1].axis("off")

axes[2].imshow(blurred, cmap="gray")
axes[2].set_title("3. Gaussian Blur (σ=1.0)")
axes[2].axis("off")

axes[3].imshow(edges, cmap="gray")
axes[3].set_title("4. Canny Edges (50/150)")
axes[3].axis("off")

plt.suptitle("Canny Edge Detection Pipeline – Sentinel-2 Glacier Tongue", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "day06_pipeline_steps.png"), dpi=150, bbox_inches="tight")
plt.show()

The pipeline preview confirms that edge detection captures structural features in the Sentinel-2 glacier image. The quality of the result depends strongly on the Gaussian blur radius and Canny threshold parameters, which will be investigated systematically in the following days.